In [132]:
import pandas as pd
import xarray as xr
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import tqdm
import xagg
xagg.set_options(silent=True)
import sys
sys.path.append('../')
from utils.text_utils import normalize_municipality_name
from utils.clim_utils import *

mun_gdf = gpd.read_file('/Volumes/Dhruv_External_Disk/portugal_borders/CAOP_Continente_2023-shp/Cont_Mun_CAOP2023.shp')
mun_gdf['concelho'] = mun_gdf['Municipio'].apply(normalize_municipality_name)
# convert crs to world mercator 84
mun_gdf = mun_gdf.to_crs(epsg=4326)


minx, miny, maxx, maxy = mun_gdf.total_bounds




In [133]:
# dissolve spatial boundaries on Distrito column to get distrio_gdf with distrito borders
distrito_gdf = mun_gdf.dissolve(by='Distrito', as_index=False)
distrito_gdf['distrito'] = distrito_gdf['Distrito'].apply(normalize_municipality_name)
# dissolve spatial boundaries on NUTSIII column to get nuts_gdf with NUTSIII borders
nuts_gdf = mun_gdf.dissolve(by='NUTSIII', as_index=False)
nuts_gdf['nutsIII'] = nuts_gdf['NUTSIII'].apply(normalize_municipality_name)


In [134]:
distrito_gdf = distrito_gdf[['distrito', 'geometry', 'DICO', 'Distrito', 'NUTSIII', 'NUTSII']].copy()
nuts_gdf = nuts_gdf[['nutsIII', 'geometry', 'NUTSIII', 'NUTSII']].copy()

In [135]:
tourist_months = [4, 5, 6, 7, 8, 9, 10] # April to October
non_tourist_months = [1, 2, 3, 11, 12] # November to March
baseline_period_1 = range(1991, 2020)
baseline_period_2 = range(1981, 2010)
baseline_period_3 = range(1971, 2019)
etcddi_baseline = range(1961, 1990)

In [136]:
# seasons
season_months = {
    'winter': [12, 1, 2],
    'spring': [3, 4, 5],
    'summer': [6, 7, 8],
    'autumn': [9, 10, 11]
}
# quarters
quarter_months = {
    'Q1': [1, 2, 3],
    'Q2': [4, 5, 6],
    'Q3': [7, 8, 9],
    'Q4': [10, 11, 12]
}


# Temperature

In [137]:
t_baseline_1961_1990_monthly_df = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/historical_baselines/portugal/monthly_baselines/t2m_baseline_1961_1990_monthly.parquet').rename(columns={'t2m_mu': 't2m_baseline_1961_1990'})
t_baseline_1971_2019_monthly_df = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/historical_baselines/portugal/monthly_baselines/t2m_baseline_1971_2019_monthly.parquet').rename(columns={'t2m_mu': 't2m_baseline_1971_2019'})
t_baseline_1981_2010_monthly_df = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/historical_baselines/portugal/monthly_baselines/t2m_baseline_1981_2010_monthly.parquet').rename(columns={'t2m_mu': 't2m_baseline_1981_2010'})
t_baseline_1991_2020_monthly_df = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/historical_baselines/portugal/monthly_baselines/t2m_baseline_1991_2020_monthly.parquet').rename(columns={'t2m_mu': 't2m_baseline_1991_2020'})

In [138]:
monthly_mean_t2m_df = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/monthly_data/portugal/t2m_monthly_portugal_avg.parquet')
monthly_mean_t2m_df = monthly_mean_t2m_df.rename(columns={'time': 'valid_time'})
monthly_mean_t2m_df['month'] = monthly_mean_t2m_df['valid_time'].dt.month
monthly_mean_t2m_df['year'] = monthly_mean_t2m_df['valid_time'].dt.year

In [139]:
t_baseline_1991_2020_monthly_df_r = t_baseline_1991_2020_monthly_df.copy()
t_baseline_1991_2020_monthly_df_r['latitude'] = t_baseline_1991_2020_monthly_df_r['latitude'].round(1)
t_baseline_1991_2020_monthly_df_r['longitude'] = t_baseline_1991_2020_monthly_df_r['longitude'].round(1)

In [140]:
# subtract monthly baseline from monthly mean to get anomalies
monthly_mean_t2m_df_r = monthly_mean_t2m_df.copy()
monthly_mean_t2m_df_r['latitude'] = monthly_mean_t2m_df_r['latitude'].round(1)
monthly_mean_t2m_df_r['longitude'] = monthly_mean_t2m_df_r['longitude'].round(1)
monthly_anomaly_1961_1990_df = monthly_mean_t2m_df_r.merge(t_baseline_1961_1990_monthly_df, on=['latitude', 'longitude', 'month'], how='left')
monthly_anomaly_1961_1990_df['t2m_anomaly_1961_1990'] = monthly_anomaly_1961_1990_df['t2m_mu'] - monthly_anomaly_1961_1990_df['t2m_baseline_1961_1990']
#-- 
monthly_anomaly_1971_2019_df = monthly_mean_t2m_df_r.merge(t_baseline_1971_2019_monthly_df, on=['latitude', 'longitude', 'month'], how='left')
monthly_anomaly_1971_2019_df['t2m_anomaly_1971_2019'] = monthly_anomaly_1971_2019_df['t2m_mu'] - monthly_anomaly_1971_2019_df['t2m_baseline_1971_2019']
#-- 
monthly_anomaly_1981_2010_df = monthly_mean_t2m_df_r.merge(t_baseline_1981_2010_monthly_df, on=['latitude', 'longitude', 'month'], how='left')
monthly_anomaly_1981_2010_df['t2m_anomaly_1981_2010'] = monthly_anomaly_1981_2010_df['t2m_mu'] - monthly_anomaly_1981_2010_df['t2m_baseline_1981_2010']
#-- 
monthly_anomaly_1991_2020_df = monthly_mean_t2m_df_r.merge(t_baseline_1991_2020_monthly_df_r, on=['latitude', 'longitude', 'month'], how='left')
monthly_anomaly_1991_2020_df['t2m_anomaly_1991_2020'] = monthly_anomaly_1991_2020_df['t2m_mu'] - monthly_anomaly_1991_2020_df['t2m_baseline_1991_2020']
#--
# merge all 4 anomaly dfs into a single df
from functools import reduce

anomalies_dfs_m_base = reduce(lambda left, right: pd.merge(left, right, on=['latitude', 'longitude', 'valid_time'], how='left'), [
    monthly_anomaly_1961_1990_df[['latitude', 'longitude', 'valid_time', 't2m_anomaly_1961_1990']],
    monthly_anomaly_1971_2019_df[['latitude', 'longitude', 'valid_time', 't2m_anomaly_1971_2019']],
    monthly_anomaly_1981_2010_df[['latitude', 'longitude', 'valid_time', 't2m_anomaly_1981_2010']],
    monthly_anomaly_1991_2020_df[['latitude', 'longitude', 'valid_time', 't2m_anomaly_1991_2020']]
])

In [141]:
t_baseline_1961_1990 = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/historical_baselines/portugal/t2m_baseline_1961_1990.parquet').rename(columns={'t2m_mu': 't2m_baseline_1961_1990'})
t_baseline_1971_2019 = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/historical_baselines/portugal/t2m_baseline_1971_2019.parquet').rename(columns={'t2m_mu': 't2m_baseline_1971_2019'})
t_baseline_1981_2010 = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/historical_baselines/portugal/t2m_baseline_1981_2010.parquet').rename(columns={'t2m_mu': 't2m_baseline_1981_2010'})
t_baseline_1991_2020 = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/historical_baselines/portugal/t2m_baseline_1991_2020.parquet').rename(columns={'t2m_mu': 't2m_baseline_1991_2020'})

In [142]:

t_baseline_1961_1990_r = round_lat_lon(t_baseline_1961_1990)
t_baseline_1971_2019_r = round_lat_lon(t_baseline_1971_2019)
t_baseline_1981_2010_r = round_lat_lon(t_baseline_1981_2010)
t_baseline_1991_2020_r = round_lat_lon(t_baseline_1991_2020)

In [143]:
monthly_std_t2m_df = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/monthly_data/portugal/t2m_monthly_portugal_std.parquet')
monthly_std_t2m_df = monthly_std_t2m_df.rename(columns={'time': 'valid_time'})
monthly_std_t2m_df['year'] = monthly_std_t2m_df['valid_time'].dt.year


In [144]:
monthly_anomaly_1961_1990_df_overall = monthly_mean_t2m_df_r.merge(t_baseline_1961_1990_r, on=['latitude', 'longitude'], how='left')
monthly_anomaly_1961_1990_df_overall['t2m_anomaly_1961_1990'] = monthly_anomaly_1961_1990_df_overall['t2m_mu'] - monthly_anomaly_1961_1990_df_overall['t2m_baseline_1961_1990']
#---
monthly_anomaly_1971_2019_df_overall = monthly_mean_t2m_df_r.merge(t_baseline_1971_2019_r, on=['latitude', 'longitude'], how='left')
monthly_anomaly_1971_2019_df_overall['t2m_anomaly_1971_2019'] = monthly_anomaly_1971_2019_df_overall['t2m_mu'] - monthly_anomaly_1971_2019_df_overall['t2m_baseline_1971_2019']
#---
monthly_anomaly_1981_2010_df_overall = monthly_mean_t2m_df_r.merge(t_baseline_1981_2010_r, on=['latitude', 'longitude'], how='left')
monthly_anomaly_1981_2010_df_overall['t2m_anomaly_1981_2010'] = monthly_anomaly_1981_2010_df_overall['t2m_mu'] - monthly_anomaly_1981_2010_df_overall['t2m_baseline_1981_2010']
#---
monthly_anomaly_1991_2020_df_overall = monthly_mean_t2m_df_r.merge(t_baseline_1991_2020_r, on=['latitude', 'longitude'], how='left')
monthly_anomaly_1991_2020_df_overall['t2m_anomaly_1991_2020'] = monthly_anomaly_1991_2020_df_overall['t2m_mu'] - monthly_anomaly_1991_2020_df_overall['t2m_baseline_1991_2020']
#-- 
# merge all 4 anomaly dfs into a single df
anomalies_dfs_overall_base = reduce(lambda left, right: pd.merge(left, right, on=['latitude', 'longitude', 'valid_time'], how='left'), [
    monthly_anomaly_1961_1990_df_overall[['latitude', 'longitude', 'valid_time', 't2m_anomaly_1961_1990']],
    monthly_anomaly_1971_2019_df_overall[['latitude', 'longitude', 'valid_time', 't2m_anomaly_1971_2019']],
    monthly_anomaly_1981_2010_df_overall[['latitude', 'longitude', 'valid_time', 't2m_anomaly_1981_2010']],
    monthly_anomaly_1991_2020_df_overall[['latitude', 'longitude', 'valid_time', 't2m_anomaly_1991_2020']]
])


In [145]:
degree_days_df = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/temperature/monthly_data/portugal/degree_days_monthly_portugal.parquet')
degree_days_df_r = degree_days_df.copy()
degree_days_df_r = round_lat_lon(degree_days_df_r)

In [146]:
def aggregate_to_region(df, region_gdf, time='year'):
    # Clean the geometry aggressively to prevent GEOS TopologyExceptions
    region_clean = region_gdf.copy()
    
    # Apply a tiny buffer in & out to eliminate micro-slivers and collinear points,
    # then simplify by ~11 meters (0.0001 deg) to avoid precision collapse, and re-validate.
    region_clean['geometry'] = (region_clean['geometry']
                                .buffer(0.0001)
                                .buffer(-0.0001)
                                .simplify(0.0001, preserve_topology=True)
                                .make_valid())
    
    # convert df to xarray dataset
    ds = xr.Dataset.from_dataframe(df.set_index(['latitude', 'longitude', time]))
    
    # define aggregation using subset_bbox=True (similar to your functional municipality logic)
    weightmap = xagg.pixel_overlaps(ds, region_clean, subset_bbox=True)
    agg = xagg.aggregate(ds, weightmap)
    agg_df = agg.to_dataframe().reset_index()
    
    # Drop poly_idx if it exists
    if 'poly_idx' in agg_df.columns:
        agg_df = agg_df.drop(columns=['poly_idx'])
        
    return agg_df

#### Distrito aggregation

In [147]:
# Process variables for Quarter Months
aggregated_quarter_dfs_distrito = []

for q_name, q_months in season_months.items():
    suffix = f"_{q_name}"
    print(f"Processing {q_name} (Months: {q_months})...")
    
    # 1. Mean Temperature
    mean_t2m = monthly_mean_t2m_df_r[monthly_mean_t2m_df_r['valid_time'].dt.month.isin(q_months)].groupby(['latitude', 'longitude', 'year'])['t2m_mu'].mean().reset_index()
    mean_t2m.rename(columns={'t2m_mu': f't2m_mu{suffix}'}, inplace=True)
    
    # 2. Temperature Variability (Std)
    # Note: Using the monthly_std_t2m_df source
    std_t2m = monthly_std_t2m_df[monthly_std_t2m_df['valid_time'].dt.month.isin(q_months)].groupby(['latitude', 'longitude', 'year'])['t2m_std'].mean().reset_index()
    std_t2m.rename(columns={'t2m_std': f't2m_std{suffix}'}, inplace=True)
    std_t2m = round_lat_lon(std_t2m)
    # 3. Anomalies (Monthly Base)
    anom_m = anomalies_dfs_m_base[anomalies_dfs_m_base['valid_time'].dt.month.isin(q_months)].copy()
    anom_m['year'] = anom_m['valid_time'].dt.year
    anom_m = anom_m.groupby(['latitude', 'longitude', 'year'])[['t2m_anomaly_1961_1990', 't2m_anomaly_1971_2019', 't2m_anomaly_1981_2010', 't2m_anomaly_1991_2020']].mean().reset_index()
    # Square columns
    anom_m = square_columns(anom_m, ['t2m_anomaly_1961_1990', 't2m_anomaly_1971_2019', 't2m_anomaly_1981_2010', 't2m_anomaly_1991_2020'])
    # Rename: suffix _m_{Q}
    anom_m.rename(columns=lambda x: f"{x}_m{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)
    
    # 4. Anomalies (Overall Base)
    anom_o = anomalies_dfs_overall_base[anomalies_dfs_overall_base['valid_time'].dt.month.isin(q_months)].copy()
    anom_o['year'] = anom_o['valid_time'].dt.year
    anom_o = anom_o.groupby(['latitude', 'longitude', 'year'])[['t2m_anomaly_1961_1990', 't2m_anomaly_1971_2019', 't2m_anomaly_1981_2010', 't2m_anomaly_1991_2020']].mean().reset_index()
    # Square columns
    anom_o = square_columns(anom_o, ['t2m_anomaly_1961_1990', 't2m_anomaly_1971_2019', 't2m_anomaly_1981_2010', 't2m_anomaly_1991_2020'])
    # Rename: suffix _o_{Q}
    anom_o.rename(columns=lambda x: f"{x}_o{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)

    # 5. Degree Days
    dd = degree_days_df_r[degree_days_df_r['valid_time'].dt.month.isin(q_months)].copy()
    dd['year'] = dd['valid_time'].dt.year
    dd = dd.groupby(['latitude', 'longitude', 'year'])[['HDD_0', 'HDD_5', 'CDD_25', 'CDD_30', 'CDD_35']].sum().reset_index()
    # Rename: suffix _dd_{Q}
    dd.rename(columns=lambda x: f"{x}_dd{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)

    # Aggregate to Municipality
    q_dfs_list = [mean_t2m, std_t2m, anom_m, anom_o, dd]
    
    for df in q_dfs_list:
        agg_df = aggregate_to_region(df, distrito_gdf)
        # Drop extra columns from shapefile merge
        #cols_to_drop = ['fid', 'DICO', 'Distrito', 'N_Freguesi', 'NUTSI', 'Alt_Max', 'Alt_Min', 'Area_ha', 'Perim_km']
        #agg_df = agg_df.drop(columns=[c for c in cols_to_drop if c in agg_df.columns])
        aggregated_quarter_dfs_distrito.append(agg_df)


#----


Processing winter (Months: [12, 1, 2])...


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are 

Processing spring (Months: [3, 4, 5])...


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are 

Processing summer (Months: [6, 7, 8])...


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are 

Processing autumn (Months: [9, 10, 11])...


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are 

In [148]:
aggregated_quarter_dfs_distrito[0]

,year,distrito,DICO,Distrito,NUTSIII,NUTSII,t2m_mu_winter
0,1950,aveiro,0101,Aveiro,Região de Aveiro,Centro,-265.246781
1,1951,aveiro,0101,Aveiro,Região de Aveiro,Centro,-264.835731
2,1952,aveiro,0101,Aveiro,Região de Aveiro,Centro,-265.312719
3,1953,aveiro,0101,Aveiro,Região de Aveiro,Centro,-264.919788
4,1954,aveiro,0101,Aveiro,Região de Aveiro,Centro,-266.258304
...,...,...,...,...,...,...,...
1363,2021,evora,0701,Évora,Alentejo Central,Alentejo,-262.283970
1364,2022,evora,0701,Évora,Alentejo Central,Alentejo,-261.140186
1365,2023,evora,0701,Évora,Alentejo Central,Alentejo,-262.989601
1366,2024,evora,0701,Évora,Alentejo Central,Alentejo,-261.359623


In [149]:
# Merge all quarterly dataframes
print("Merging quarterly data...")
final_aggregated_quarter_df = reduce(lambda left, right: pd.merge(left, right, on=['year', 'distrito', 'Distrito', 'DICO', 'NUTSII', 'NUTSIII'], how='left'), aggregated_quarter_dfs_distrito)

# # Update the final aggregated dataframe
# final_aggregated_df = pd.merge(final_aggregated_df, final_aggregated_quarter_df, on=['concelho', 'year', 'Municipio', 'NUTSIII', 'NUTSII'], how='left')

print("Done. New columns added:", len(final_aggregated_quarter_df.columns) - 5)

Merging quarterly data...
Done. New columns added: 93


In [150]:
final_aggregated_quarter_df

,year,distrito,DICO,Distrito,NUTSIII,NUTSII,t2m_mu_winter,t2m_std_winter,t2m_anomaly_1961_1990_m_winter,t2m_anomaly_1971_2019_m_winter,...,t2m_anomaly_1991_2020_o_autumn,t2m_anomaly_1961_1990_squared_o_autumn,t2m_anomaly_1971_2019_squared_o_autumn,t2m_anomaly_1981_2010_squared_o_autumn,t2m_anomaly_1991_2020_squared_o_autumn,HDD_0_dd_autumn,HDD_5_dd_autumn,CDD_25_dd_autumn,CDD_30_dd_autumn,CDD_35_dd_autumn
0,1950,aveiro,0101,Aveiro,Região de Aveiro,Centro,-265.246781,2.112789,-273.850824,-273.932555,...,-271.902948,73625.839414,73765.103778,73870.250595,73931.225288,0.0,0.000000,0.000000,0.0,0.0
1,1951,aveiro,0101,Aveiro,Região de Aveiro,Centro,-264.835731,1.979116,-273.439780,-273.521507,...,-273.219957,74342.276242,74482.215858,74587.882900,74649.154250,0.0,0.000000,0.000000,0.0,0.0
2,1952,aveiro,0101,Aveiro,Região de Aveiro,Centro,-265.312719,2.076245,-273.916766,-273.998494,...,-273.155703,74307.240328,74447.147656,74552.790776,74614.048611,0.0,0.819052,0.000000,0.0,0.0
3,1953,aveiro,0101,Aveiro,Região de Aveiro,Centro,-264.919788,2.362185,-273.523832,-273.605563,...,-272.175889,73774.012383,73913.417916,74018.679477,74079.720250,0.0,0.000000,2.853906,0.0,0.0
4,1954,aveiro,0101,Aveiro,Região de Aveiro,Centro,-266.258304,2.745991,-274.862343,-274.944074,...,-271.678984,73504.346620,73643.492462,73748.553460,73809.479776,0.0,0.001749,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1363,2021,evora,0701,Évora,Alentejo Central,Alentejo,-262.283970,2.330188,-271.909333,-272.008614,...,-272.083325,73640.125224,73806.880916,73907.046838,74029.346637,0.0,0.000000,3.681894,0.0,0.0
1364,2022,evora,0701,Évora,Alentejo Central,Alentejo,-261.140186,1.499218,-270.765548,-270.864830,...,-271.054403,73082.774957,73248.886087,73348.666740,73470.495532,0.0,0.000000,2.146368,0.0,0.0
1365,2023,evora,0701,Évora,Alentejo Central,Alentejo,-262.989601,2.435673,-272.614964,-272.714247,...,-270.962760,73033.213153,73199.278566,73299.027574,73420.823176,0.0,0.000000,11.785569,0.0,0.0
1366,2024,evora,0701,Évora,Alentejo Central,Alentejo,-261.359623,2.353038,-270.984984,-271.084265,...,-271.379601,73258.685044,73425.005568,73524.911728,73646.898393,0.0,0.000000,7.265134,0.0,0.0


#### NUTS3 aggregation

In [151]:
# Process variables for Quarter Months
aggregated_quarter_dfs_nuts = []

for q_name, q_months in season_months.items():
    suffix = f"_{q_name}"
    print(f"Processing {q_name} (Months: {q_months})...")
    
    # 1. Mean Temperature
    mean_t2m = monthly_mean_t2m_df_r[monthly_mean_t2m_df_r['valid_time'].dt.month.isin(q_months)].groupby(['latitude', 'longitude', 'year'])['t2m_mu'].mean().reset_index()
    mean_t2m.rename(columns={'t2m_mu': f't2m_mu{suffix}'}, inplace=True)
    
    # 2. Temperature Variability (Std)
    # Note: Using the monthly_std_t2m_df source
    std_t2m = monthly_std_t2m_df[monthly_std_t2m_df['valid_time'].dt.month.isin(q_months)].groupby(['latitude', 'longitude', 'year'])['t2m_std'].mean().reset_index()
    std_t2m.rename(columns={'t2m_std': f't2m_std{suffix}'}, inplace=True)
    std_t2m = round_lat_lon(std_t2m)
    # 3. Anomalies (Monthly Base)
    anom_m = anomalies_dfs_m_base[anomalies_dfs_m_base['valid_time'].dt.month.isin(q_months)].copy()
    anom_m['year'] = anom_m['valid_time'].dt.year
    anom_m = anom_m.groupby(['latitude', 'longitude', 'year'])[['t2m_anomaly_1961_1990', 't2m_anomaly_1971_2019', 't2m_anomaly_1981_2010', 't2m_anomaly_1991_2020']].mean().reset_index()
    # Square columns
    anom_m = square_columns(anom_m, ['t2m_anomaly_1961_1990', 't2m_anomaly_1971_2019', 't2m_anomaly_1981_2010', 't2m_anomaly_1991_2020'])
    # Rename: suffix _m_{Q}
    anom_m.rename(columns=lambda x: f"{x}_m{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)
    
    # 4. Anomalies (Overall Base)
    anom_o = anomalies_dfs_overall_base[anomalies_dfs_overall_base['valid_time'].dt.month.isin(q_months)].copy()
    anom_o['year'] = anom_o['valid_time'].dt.year
    anom_o = anom_o.groupby(['latitude', 'longitude', 'year'])[['t2m_anomaly_1961_1990', 't2m_anomaly_1971_2019', 't2m_anomaly_1981_2010', 't2m_anomaly_1991_2020']].mean().reset_index()
    # Square columns
    anom_o = square_columns(anom_o, ['t2m_anomaly_1961_1990', 't2m_anomaly_1971_2019', 't2m_anomaly_1981_2010', 't2m_anomaly_1991_2020'])
    # Rename: suffix _o_{Q}
    anom_o.rename(columns=lambda x: f"{x}_o{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)

    # 5. Degree Days
    dd = degree_days_df_r[degree_days_df_r['valid_time'].dt.month.isin(q_months)].copy()
    dd['year'] = dd['valid_time'].dt.year
    dd = dd.groupby(['latitude', 'longitude', 'year'])[['HDD_0', 'HDD_5', 'CDD_25', 'CDD_30', 'CDD_35']].sum().reset_index()
    # Rename: suffix _dd_{Q}
    dd.rename(columns=lambda x: f"{x}_dd{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)

    # Aggregate to NUTS3
    q_dfs_list = [mean_t2m, std_t2m, anom_m, anom_o, dd]
    
    for df in q_dfs_list:
        agg_df = aggregate_to_region(df, nuts_gdf)
        # Drop extra columns from shapefile merge
        #cols_to_drop = ['fid', 'DICO', 'Distrito', 'N_Freguesi', 'NUTSI', 'Alt_Max', 'Alt_Min', 'Area_ha', 'Perim_km']
        #agg_df = agg_df.drop(columns=[c for c in cols_to_drop if c in agg_df.columns])
        aggregated_quarter_dfs_nuts.append(agg_df)


#----


Processing winter (Months: [12, 1, 2])...


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are 

Processing spring (Months: [3, 4, 5])...


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are 

Processing summer (Months: [6, 7, 8])...


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are 

Processing autumn (Months: [9, 10, 11])...


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are 

In [152]:
aggregated_quarter_dfs_nuts[3]

,year,nutsIII,NUTSIII,NUTSII,t2m_anomaly_1961_1990_o_winter,t2m_anomaly_1971_2019_o_winter,t2m_anomaly_1981_2010_o_winter,t2m_anomaly_1991_2020_o_winter,t2m_anomaly_1961_1990_squared_o_winter,t2m_anomaly_1971_2019_squared_o_winter,t2m_anomaly_1981_2010_squared_o_winter,t2m_anomaly_1991_2020_squared_o_winter
0,1950,alentejo central,Alentejo Central,Alentejo,-280.145994,-280.453070,-280.637348,-280.862188,78481.910120,78654.079815,78757.494308,78883.761079
1,1951,alentejo central,Alentejo Central,Alentejo,-279.866253,-280.173327,-280.357609,-280.582446,78325.261485,78497.258517,78600.571675,78726.712805
2,1952,alentejo central,Alentejo Central,Alentejo,-280.404563,-280.711637,-280.895920,-281.120756,78626.856677,78799.184633,78902.697399,79029.079724
3,1953,alentejo central,Alentejo Central,Alentejo,-279.793091,-280.100166,-280.284448,-280.509285,78284.344056,78456.300233,78559.588363,78685.698039
4,1954,alentejo central,Alentejo Central,Alentejo,-280.959075,-281.266151,-281.450430,-281.675269,78938.102223,79110.768056,79214.480863,79341.110885
...,...,...,...,...,...,...,...,...,...,...,...,...
1819,2021,area metropolitana do porto,Área Metropolitana do Porto,Norte,-277.154364,-277.433000,-277.629219,-277.757692,76814.585475,76969.111936,77078.037520,77149.389341
1820,2022,area metropolitana do porto,Área Metropolitana do Porto,Norte,-276.587270,-276.865910,-277.062126,-277.190599,76500.545773,76654.757175,76763.454420,76834.660814
1821,2023,area metropolitana do porto,Área Metropolitana do Porto,Norte,-277.941035,-278.219680,-278.415892,-278.544366,77251.342899,77406.313043,77515.551840,77587.104579
1822,2024,area metropolitana do porto,Área Metropolitana do Porto,Norte,-276.480204,-276.758838,-276.955058,-277.083528,76441.347299,76595.495764,76704.156491,76775.334414


In [153]:
# Merge all quarterly dataframes
print("Merging quarterly data...")
final_aggregated_quarter_df_nuts = reduce(lambda left, right: pd.merge(left, right, on=['year', 'nutsIII', 'NUTSII', 'NUTSIII'], how='left'), aggregated_quarter_dfs_nuts)

# # Update the final aggregated dataframe
# final_aggregated_df = pd.merge(final_aggregated_df, final_aggregated_quarter_df, on=['concelho', 'year', 'Municipio', 'NUTSIII', 'NUTSII'], how='left')

print("Done. New columns added:", len(final_aggregated_quarter_df_nuts.columns) - 5)

Merging quarterly data...
Done. New columns added: 91


# Rain

In [62]:
ex_tp_total_1961_1990 = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/precipitation/historical/monthly/portugal/extreme_tp_tot_kotz_1961_1989.parquet')
ex_tp_total_1991_2020 = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/precipitation/historical/monthly/portugal/extreme_tp_tot_kotz_1991_2019.parquet')
ex_tp_total_1981_2009 = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/precipitation/historical/monthly/portugal/extreme_tp_tot_kotz_1981_2009.parquet')
ex_tp_total_1971_2019 = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/precipitation/historical/monthly/portugal/extreme_tp_tot_kotz_1971_2018.parquet')


In [63]:
ex_tp_total_1961_1990_r = round_lat_lon(ex_tp_total_1961_1990)
ex_tp_total_1991_2020_r = round_lat_lon(ex_tp_total_1991_2020)
ex_tp_total_1981_2009_r = round_lat_lon(ex_tp_total_1981_2009)
ex_tp_total_1971_2019_r = round_lat_lon(ex_tp_total_1971_2019)


In [64]:
# filter for year after 1979 inclusive and before 2019 inclusive
ex_tp_total_1961_1990_r = ex_tp_total_1961_1990_r[(ex_tp_total_1961_1990_r['month'].dt.year >= 1979) & (ex_tp_total_1961_1990_r['month'].dt.year <= 2025)]
ex_tp_total_1991_2020_r = ex_tp_total_1991_2020_r[(ex_tp_total_1991_2020_r['month'].dt.year >= 1979) & (ex_tp_total_1991_2020_r['month'].dt.year <= 2025)]
ex_tp_total_1981_2009_r = ex_tp_total_1981_2009_r[(ex_tp_total_1981_2009_r['month'].dt.year >= 1979) & (ex_tp_total_1981_2009_r['month'].dt.year <= 2025)]
ex_tp_total_1971_2019_r = ex_tp_total_1971_2019_r[(ex_tp_total_1971_2019_r['month'].dt.year >= 1979) & (ex_tp_total_1971_2019_r['month'].dt.year <= 2025)]
#----
# add suffixes
ex_tp_total_1961_1990_r = ex_tp_total_1961_1990_r.rename(columns=lambda x: f"{x}_base61_90" if x not in ['latitude', 'longitude', 'month'] else x)
ex_tp_total_1991_2020_r = ex_tp_total_1991_2020_r.rename(columns=lambda x: f"{x}_base91_20" if x not in ['latitude', 'longitude', 'month'] else x)
ex_tp_total_1981_2009_r = ex_tp_total_1981_2009_r.rename(columns=lambda x: f"{x}_base81_09" if x not in ['latitude', 'longitude', 'month'] else x)
ex_tp_total_1971_2019_r = ex_tp_total_1971_2019_r.rename(columns=lambda x: f"{x}_base71_19" if x not in ['latitude', 'longitude', 'month'] else x)

In [65]:
tp_monthly_tot = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/precipitation/historical/monthly/portugal/tp_monthly_portugal_total.parquet')
tp_monthly_tot_r = round_lat_lon(tp_monthly_tot)
tp_monthly_tot_r['year'] = tp_monthly_tot_r['valid_time'].dt.year


In [66]:
wet_day_df = pd.read_parquet(r'/Volumes/Dhruv_External_Disk/climate_data/precipitation/historical/monthly/portugal/monthly_wet_days_portugal.parquet')
wet_day_df_r = round_lat_lon(wet_day_df)
wet_day_df_r['year'] = wet_day_df_r['valid_time'].dt.year


In [67]:
baselines = {
    '1991_2020': baseline_period_1,
    '1981_2010': baseline_period_2,
    '1971_2019': baseline_period_3,
    '1961_1990': etcddi_baseline
}

In [82]:
def compute_kotz_indices(region_month_df, baseline_range, region_col='concelho'):
    # Prepare data
    df = region_month_df.copy()
    
    # 2. Baseline month mean (mu) and dispersion (sigma)
    baseline_df = df[df['year'].isin(baseline_range)]
    baseline_stats = baseline_df.groupby([region_col, 'month'])['tp_total'].agg(['mean', 'std']).reset_index()
    baseline_stats.rename(columns={'mean': 'mu', 'std': 'sigma'}, inplace=True)
    
    # 3. Baseline mean annual total (RA)
    # Sum of mu across all 12 months for each region
    RA_stats = baseline_stats.groupby(region_col)['mu'].sum().reset_index()
    RA_stats.rename(columns={'mu': 'RA'}, inplace=True)
    
    # Merge stats back to the full dataset
    df = pd.merge(df, baseline_stats, on=[region_col, 'month'], how='left')
    df = pd.merge(df, RA_stats, on=[region_col], how='left')
    
    # 4. Seasonal standardized monthly deviations term
    # term = ((R - mu) / sigma) * (mu / RA)
    # Handle sigma=0 if necessary, though unlikely for precipitation unless perfectly dry baseline
    df['deviation_term'] = ((df['tp_total'] - df['mu']) / df['sigma']) * (df['mu'] / df['RA'])
    
    # Sum across seasons
    # Tourist Season

    return df

In [69]:
def aggregate_to_municipality(df, mun_gdf, time = 'valid_time'):
    # Create xarray dataset from dataframe
    # Ensure we use valid_time to preserve monthly resolution
    ds = df.set_index(['latitude', 'longitude', time]).to_xarray()
    
    # Calculate weight map for aggregation
    weightmap = xagg.pixel_overlaps(ds, mun_gdf, subset_bbox=True)
    
    # Aggregate (calculates area-weighted mean by default)
    agg = xagg.aggregate(ds, weightmap)
    
    # Convert back to dataframe
    agg_df = agg.to_dataframe().reset_index()
    
    # Map poly_idx back to concelho names
    # Assuming mun_gdf has a default index or one that matches poly_idx 0..N
    agg_df['concelho'] = agg_df['poly_idx'].map(lambda x: mun_gdf.iloc[x]['concelho'])
    
    if 'poly_idx' in agg_df.columns:
        agg_df = agg_df.drop(columns=['poly_idx'])
        
    return agg_df
tp_region_month = aggregate_to_municipality(tp_monthly_tot_r, mun_gdf)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total have inconsistent nans along the dimension(s) valid_time (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along valid_time. The aggregation calculation may therefore be incorrect, since it aggregates over different grid cells for  the same polgyon for different coordinates of the dimension(s)valid_time.
  warnings.warn('One or more grid cells in variable '+var+' have inconsistent nans along the dimension(s) '+


In [89]:
# ...existing code...
# Process variables for Quarter Months (Rainfall)
print("Processing quarterly rainfall variables...")

from functools import reduce

aggregated_quarter_dfs_rain_distrito = []
kotz_quarter_results = []

# List of Extreme Precip DFs
ex_dfs_list = [ex_tp_total_1961_1990_r, ex_tp_total_1991_2020_r, ex_tp_total_1981_2009_r, ex_tp_total_1971_2019_r]

# Ensure tp_region_month has 'year' and 'month' attributes
if 'valid_time' in tp_region_month.columns:
    if 'year' not in tp_region_month.columns:
        tp_region_month['year'] = tp_region_month['valid_time'].dt.year
    if 'month' not in tp_region_month.columns:
        tp_region_month['month'] = tp_region_month['valid_time'].dt.month

for q_name, q_months in season_months.items():
    suffix = f"_{q_name}"
    print(f"  {q_name}: {q_months}")

    # 1. Total & Mean Precip (Grid)
    # tp_monthly_tot_r has 'valid_time'
    sub_tp = tp_monthly_tot_r[tp_monthly_tot_r['valid_time'].dt.month.isin(q_months)].copy()
    sub_tp['year'] = sub_tp['valid_time'].dt.year
    
    # Sum
    q_tp_sum = sub_tp.groupby(['latitude', 'longitude', 'year'])['tp_total'].sum().reset_index()
    q_tp_sum.rename(columns={'tp_total': f'tp_total{suffix}'}, inplace=True)
    
    # Mean
    q_tp_mean = sub_tp.groupby(['latitude', 'longitude', 'year'])['tp_total'].mean().reset_index()
    q_tp_mean.rename(columns={'tp_total': f'tp_mean{suffix}'}, inplace=True)
    
    # 2. Wet Days (Grid)
    # wet_day_df_r has 'valid_time'
    sub_wd = wet_day_df_r[wet_day_df_r['valid_time'].dt.month.isin(q_months)].copy()
    sub_wd['year'] = sub_wd['valid_time'].dt.year
    q_wd_sum = sub_wd.groupby(['latitude', 'longitude', 'year'])['wet_days'].sum().reset_index()
    q_wd_sum.rename(columns={'wet_days': f'wet_days{suffix}'}, inplace=True)
    
    # 3. Extreme Precip (Grid)
    q_ex_sum_list = []
    q_ex_avg_list = []
    
    for df_ex in ex_dfs_list:
        # Filter month
        sub_ex = df_ex[df_ex['month'].dt.month.isin(q_months)].copy()
        sub_ex['year'] = sub_ex['month'].dt.year
        
        # Sum
        s_sum = sub_ex.drop(columns=['month']).groupby(['latitude', 'longitude', 'year']).sum().reset_index()
        # Rename columns to add suffix
        s_sum.rename(columns=lambda x: f"{x}{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)
        q_ex_sum_list.append(s_sum)
        
        # Mean
        s_avg = sub_ex.drop(columns=['month']).groupby(['latitude', 'longitude', 'year']).mean().reset_index()
        # Rename columns to add _avg and suffix
        s_avg.rename(columns=lambda x: f"{x}_avg{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)
        q_ex_avg_list.append(s_avg)
        
    # Merge Extreme Sums
    q_ex_sum_merged = reduce(lambda left, right: pd.merge(left, right, on=['latitude', 'longitude', 'year'], how='outer'), q_ex_sum_list)
    # Merge Extreme Means
    q_ex_avg_merged = reduce(lambda left, right: pd.merge(left, right, on=['latitude', 'longitude', 'year'], how='outer'), q_ex_avg_list)
    
    # 4. Kotz Indices (distrito directly)
    for label, period in baselines.items():
        # Compute baseline
        res = compute_kotz_indices(tp_region_month, period, region_col='Distrito')
        # Extract specific quarter using matching months
        res_q = res[res['month'].isin(q_months)]
        res_sum = res_q.groupby(['Distrito', 'year'])['deviation_term'].sum().reset_index()
        # Rename sum as RM_tourist
        res_sum = res_sum.rename(columns={'deviation_term': f'RM_tourist_{label}{suffix}'})
        kotz_quarter_results.append(res_sum)

    # 5. Aggregate Grid DFs to Municipality
    # Ensure all have integer year
    to_agg = [q_tp_sum, q_tp_mean, q_wd_sum, q_ex_sum_merged, q_ex_avg_merged]
    for i, df in enumerate(to_agg):
         # Ensure lat/lon are rounded to avoid mismatch during xagg overlap (though they should be coming from rounded parents)
        df = round_lat_lon(df)
        agg = aggregate_to_region(df, distrito_gdf, time='year')
        aggregated_quarter_dfs_rain_distrito.append(agg)

Processing quarterly rainfall variables...
  winter: [12, 1, 2]


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total_winter have inconsistent nans along the dimension(s) year (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along year. The aggregation calculation may 

  spring: [3, 4, 5]


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total_spring have inconsistent nans along the dimension(s) year (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along year. The aggregation calculation may 

  summer: [6, 7, 8]


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total_summer have inconsistent nans along the dimension(s) year (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along year. The aggregation calculation may 

  autumn: [9, 10, 11]


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total_autumn have inconsistent nans along the dimension(s) year (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along year. The aggregation calculation may 

In [ ]:
# Merge all new quarter DataFrames
print("Merging quarterly rain data...")
kotz_quarter_results = [df.merge(distrito_gdf[['NUTSIII','NUTSII','distrito', "DICO", "Distrito"]], on='Distrito', how='left') for df in kotz_quarter_results]
kotz_quarter_results = [df.groupby(['NUTSIII','NUTSII','distrito', "DICO", "Distrito", 'year']).mean().reset_index() for df in kotz_quarter_results]

all_quarter_dfs = aggregated_quarter_dfs_rain_distrito + kotz_quarter_results
for df in all_quarter_dfs:
    df['nuts3'] = df['NUTSIII'].map(normalize_municipality_name)
# Use outer join to keep all years, then we filter
final_quarter_rain = reduce(lambda left, right: pd.merge(left, right, on=['year', 'NUTSIII', 'NUTSII', 'distrito', "DICO", "Distrito", 'nuts3'], how='outer'), all_quarter_dfs)

# Filter for the relevant year range (1979-2019) to clean up NaNs from non-overlapping periods
final_quarter_rain = final_quarter_rain[(final_quarter_rain['year'] >= 1979) & (final_quarter_rain['year'] <= 2025)]

# Clean up any residual NaNs (fill with 0 if appropriate, or keep as NaN?)
# Typically rainfall totals can be 0, but indices might be NaN. 
# For now, let's just inspect it.

print("Done. Columns added:", len(final_quarter_rain.columns) - 2)


# Merge into main DF (optional, if you want to keep them separate for now, comment out)
# final_aggregated_df = pd.merge(final_aggregated_df, final_quarter_rain, on=['concelho', 'year'], how='left')
# final_aggregated_df.to_parquet(r'../../data/weather/tp_aggregated_municipality.parquet')

Done. Columns added: 97


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/3790872148.py:10: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  final_quarter_rain = reduce(lambda left, right: pd.merge(left, right, on=['year', 'NUTSIII', 'NUTSII', 'distrito', "DICO", "Distrito", 'nuts3'], how='outer'), all_quarter_dfs)


In [98]:
# ...existing code...
# Process variables for Quarter Months (Rainfall)
print("Processing quarterly rainfall variables...")

from functools import reduce

aggregated_quarter_dfs_rain_nuts3 = []
kotz_quarter_results = []

# List of Extreme Precip DFs
ex_dfs_list = [ex_tp_total_1961_1990_r, ex_tp_total_1991_2020_r, ex_tp_total_1981_2009_r, ex_tp_total_1971_2019_r]

# Ensure tp_region_month has 'year' and 'month' attributes
if 'valid_time' in tp_region_month.columns:
    if 'year' not in tp_region_month.columns:
        tp_region_month['year'] = tp_region_month['valid_time'].dt.year
    if 'month' not in tp_region_month.columns:
        tp_region_month['month'] = tp_region_month['valid_time'].dt.month

for q_name, q_months in season_months.items():
    suffix = f"_{q_name}"
    print(f"  {q_name}: {q_months}")

    # 1. Total & Mean Precip (Grid)
    # tp_monthly_tot_r has 'valid_time'
    sub_tp = tp_monthly_tot_r[tp_monthly_tot_r['valid_time'].dt.month.isin(q_months)].copy()
    sub_tp['year'] = sub_tp['valid_time'].dt.year
    
    # Sum
    q_tp_sum = sub_tp.groupby(['latitude', 'longitude', 'year'])['tp_total'].sum().reset_index()
    q_tp_sum.rename(columns={'tp_total': f'tp_total{suffix}'}, inplace=True)
    
    # Mean
    q_tp_mean = sub_tp.groupby(['latitude', 'longitude', 'year'])['tp_total'].mean().reset_index()
    q_tp_mean.rename(columns={'tp_total': f'tp_mean{suffix}'}, inplace=True)
    
    # 2. Wet Days (Grid)
    # wet_day_df_r has 'valid_time'
    sub_wd = wet_day_df_r[wet_day_df_r['valid_time'].dt.month.isin(q_months)].copy()
    sub_wd['year'] = sub_wd['valid_time'].dt.year
    q_wd_sum = sub_wd.groupby(['latitude', 'longitude', 'year'])['wet_days'].sum().reset_index()
    q_wd_sum.rename(columns={'wet_days': f'wet_days{suffix}'}, inplace=True)
    
    # 3. Extreme Precip (Grid)
    q_ex_sum_list = []
    q_ex_avg_list = []
    
    for df_ex in ex_dfs_list:
        # Filter month
        sub_ex = df_ex[df_ex['month'].dt.month.isin(q_months)].copy()
        sub_ex['year'] = sub_ex['month'].dt.year
        
        # Sum
        s_sum = sub_ex.drop(columns=['month']).groupby(['latitude', 'longitude', 'year']).sum().reset_index()
        # Rename columns to add suffix
        s_sum.rename(columns=lambda x: f"{x}{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)
        q_ex_sum_list.append(s_sum)
        
        # Mean
        s_avg = sub_ex.drop(columns=['month']).groupby(['latitude', 'longitude', 'year']).mean().reset_index()
        # Rename columns to add _avg and suffix
        s_avg.rename(columns=lambda x: f"{x}_avg{suffix}" if x not in ['latitude', 'longitude', 'year'] else x, inplace=True)
        q_ex_avg_list.append(s_avg)
        
    # Merge Extreme Sums
    q_ex_sum_merged = reduce(lambda left, right: pd.merge(left, right, on=['latitude', 'longitude', 'year'], how='outer'), q_ex_sum_list)
    # Merge Extreme Means
    q_ex_avg_merged = reduce(lambda left, right: pd.merge(left, right, on=['latitude', 'longitude', 'year'], how='outer'), q_ex_avg_list)
    
    # 4. Kotz Indices (Municipality directly)
    for label, period in baselines.items():
        # Compute baseline
        res = compute_kotz_indices(tp_region_month, period, region_col='NUTSIII')
        # Extract specific quarter using matching months
        res_q = res[res['month'].isin(q_months)]
        res_sum = res_q.groupby(['NUTSIII', 'year'])['deviation_term'].sum().reset_index()
        # Rename sum as RM_tourist
        res_sum = res_sum.rename(columns={'deviation_term': f'RM_tourist_{label}{suffix}'})
        kotz_quarter_results.append(res_sum)

    # 5. Aggregate Grid DFs to Municipality
    # Ensure all have integer year
    to_agg = [q_tp_sum, q_tp_mean, q_wd_sum, q_ex_sum_merged, q_ex_avg_merged]
    for i, df in enumerate(to_agg):
         # Ensure lat/lon are rounded to avoid mismatch during xagg overlap (though they should be coming from rounded parents)
        df = round_lat_lon(df)
        agg = aggregate_to_region(df, nuts_gdf, time='year')
        aggregated_quarter_dfs_rain_nuts3.append(agg)

Processing quarterly rainfall variables...
  winter: [12, 1, 2]


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total_winter have inconsistent nans along the dimension(s) year (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along year. The aggregation calculation may 

  spring: [3, 4, 5]


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total_spring have inconsistent nans along the dimension(s) year (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along year. The aggregation calculation may 

  summer: [6, 7, 8]


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total_summer have inconsistent nans along the dimension(s) year (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along year. The aggregation calculation may 

  autumn: [9, 10, 11]


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(0.0001)
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/306178984.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .buffer(-0.0001)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/xagg/auxfuncs.py:167: SomeNanWarning: One or more grid cells in variable tp_total_autumn have inconsistent nans along the dimension(s) year (i.e., one or more grid cells are nan for some but not all coordinates of the dimension(s)). This means that grid cell weights will be different for different coordinates along year. The aggregation calculation may 

In [107]:
# Merge all new quarter DataFrames
print("Merging quarterly rain data...")
kotz_quarter_results = [df.merge(nuts_gdf[['NUTSIII','NUTSII', 'nutsIII']], on='NUTSIII', how='left') for df in kotz_quarter_results]
kotz_quarter_results = [df.groupby(['NUTSIII','NUTSII','nutsIII','year']).mean().reset_index() for df in kotz_quarter_results]

all_quarter_dfs = aggregated_quarter_dfs_rain_nuts3 + kotz_quarter_results

# Use outer join to keep all years, then we filter
final_quarter_rain_nuts3 = reduce(lambda left, right: pd.merge(left, right, on=['year', 'NUTSIII', 'NUTSII', 'nutsIII'], how='outer'), all_quarter_dfs)

# Filter for the relevant year range (1979-2019) to clean up NaNs from non-overlapping periods
final_quarter_rain_nuts3 = final_quarter_rain_nuts3[(final_quarter_rain_nuts3['year'] >= 1979) & (final_quarter_rain_nuts3['year'] <= 2025)]

# Clean up any residual NaNs (fill with 0 if appropriate, or keep as NaN?)
# Typically rainfall totals can be 0, but indices might be NaN. 
# For now, let's just inspect it.

print("Done. Columns added:", len(final_quarter_rain_nuts3.columns) - 2)


# Merge into main DF (optional, if you want to keep them separate for now, comment out)
# final_aggregated_df = pd.merge(final_aggregated_df, final_quarter_rain, on=['concelho', 'year'], how='left')
# final_aggregated_df.to_parquet(r'../../data/weather/tp_aggregated_municipality.parquet')

Merging quarterly rain data...
Done. Columns added: 94


/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/24285312.py:9: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  final_quarter_rain_nuts3 = reduce(lambda left, right: pd.merge(left, right, on=['year', 'NUTSIII', 'NUTSII', 'nutsIII'], how='outer'), all_quarter_dfs)


___

In [154]:
final_district = final_quarter_rain.merge(final_aggregated_quarter_df, on=['year', 'NUTSIII', 'NUTSII', 'distrito', "DICO", "Distrito"], how='inner')
final_nuts3 = final_quarter_rain_nuts3.merge(final_aggregated_quarter_df_nuts, on=['year', 'NUTSIII', 'NUTSII', 'nutsIII'], how='inner')



/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/166167476.py:1: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  final_district = final_quarter_rain.merge(final_aggregated_quarter_df, on=['year', 'NUTSIII', 'NUTSII', 'distrito', "DICO", "Distrito"], how='inner')
/var/folders/nt/03y4p9md50gblp_0svv74zb80000gn/T/ipykernel_22074/166167476.py:2: UserWarning: You are merging on int and float columns where the float values are not equal to their int representation.
  final_nuts3 = final_quarter_rain_nuts3.merge(final_aggregated_quarter_df_nuts, on=['year', 'NUTSIII', 'NUTSII', 'nutsIII'], how='inner')


In [155]:
# count of null values per column in final_district
null_counts_district = final_district.isnull().sum()
print("Null counts in final_district:")
null_counts_district.sort_values(ascending=False)


Null counts in final_district:


year                                      0
t2m_anomaly_1991_2020_squared_m_spring    0
t2m_mu_spring                             0
t2m_std_spring                            0
t2m_anomaly_1961_1990_m_spring            0
                                         ..
wet_days_autumn                           0
extreme_tp_tot_p99_base61_90_autumn       0
extreme_tp_tot_p999_base61_90_autumn      0
extreme_tp_tot_p99_base91_20_autumn       0
CDD_35_dd_autumn                          0
Length: 191, dtype: int64

In [156]:
final_district.to_parquet(r'quarterly_stratified_yearly_district.parquet')
final_nuts3.to_parquet(r'quarterly_stratified_yearly_nuts3.parquet')